# Personal Budget Assistant — an Agent, not a Chatbot
**CSE476 CA1 · Topic T1**

This notebook is the proof that the project is an **agent**:

1. **It calls tools** — `add_expense` and `get_summary`, not just text.
2. **It takes more than one step** — a plan-act loop that reads each tool result and decides the next step.
3. **It remembers earlier turns** — a `Memory` object holds the running expense list + monthly budget, and a *later* turn ("can I afford …?") uses facts from an *earlier* turn.

The agent has two lanes behind the **same loop**:
- `mode="llm"` — an LLM on GitHub Models decides the steps via function-calling (needs `GITHUB_TOKEN`).
- `mode="rule"` — a deterministic planner decides the steps. **No API key**, so this notebook always runs.

`mode="auto"` (the default) picks `llm` if `GITHUB_TOKEN` is set, else `rule`.

In [1]:
from budget_agent import Agent, Memory
from budget_agent.tools import TOOL_SCHEMAS, add_expense, get_summary

print("tools the agent can call:", [t["function"]["name"] for t in TOOL_SCHEMAS])

tools the agent can call: ['add_expense', 'get_summary']


## 1. The two tools (called directly here, just to show what they do)

Normally the **agent** calls these — never the user. Each one reads/writes the shared `Memory`.

In [2]:
demo_mem = Memory()
demo_mem.monthly_budget = 10000

print(add_expense(demo_mem, item="chicken biryani", amount=220))
print(add_expense(demo_mem, item="metro card recharge", amount=150))
print(get_summary(demo_mem, category="all"))
print(get_summary(demo_mem, category="food"))

{'ok': True, 'recorded': {'item': 'chicken biryani', 'amount': 220.0, 'category': 'other'}, 'category_total': 220.0, 'total_spent': 220.0, 'balance': 9780.0}
{'ok': True, 'recorded': {'item': 'metro card recharge', 'amount': 150.0, 'category': 'transport'}, 'category_total': 150.0, 'total_spent': 370.0, 'balance': 9630.0}
{'ok': True, 'scope': 'all', 'total_spent': 370.0, 'budget': 10000, 'balance': 9630.0, 'by_category': {'other': 220.0, 'transport': 150.0}}
{'ok': True, 'scope': 'food', 'total_spent': 370.0, 'budget': 10000, 'balance': 9630.0, 'by_category': {'other': 220.0, 'transport': 150.0}, 'category_total': 0, 'count_in_category': 0}


## 2. Memory

`Memory` keeps **turn history** (every message this session) and a **structured store**
(expenses + budget). The structured store is what a later answer reads back.

In [3]:
print("turn history :", demo_mem.turns)
print("expenses     :", demo_mem.expenses)
print("snapshot     :", demo_mem.snapshot())

turn history : []
expenses     : [Expense(item='chicken biryani', amount=220.0, category='other'), Expense(item='metro card recharge', amount=150.0, category='transport')]
snapshot     : budget=10000 | spent=370 | n_expenses=2 | by_category=other:220, transport:150


## 3. Demo A — log spending, then ask an affordability question

One `Agent` instance = one conversation. Watch the trace: the agent logs two
expenses (two tool calls), then for the affordability turn it calls `get_summary`,
**reads the `balance` from that result**, subtracts 2000, and decides.

In [4]:
agent = Agent(mode="rule")   # deterministic lane so the notebook always runs
print("resolved mode:", agent.mode)

for goal in [
    "My monthly budget is 15000. I spent 250 on lunch and 400 on an uber ride.",
    "Also paid 1200 for groceries and a movie cost 300.",
    "Can I afford a 2000 trip this weekend?",
]:
    print("\nUSER:", goal)
    result = agent.run(goal)
    result.show_trace()
    print("AGENT:", result.answer)

resolved mode: rule

USER: My monthly budget is 15000. I spent 250 on lunch and 400 on an uber ride.
[step 1] MEMORY  stored monthly_budget = 15000
[step 2] THINK   user mentioned spending 250 on lunch; log it
          ACT     add_expense({'item': 'lunch', 'amount': 250.0})
          OBS     {'ok': True, 'recorded': {'item': 'lunch', 'amount': 250.0, 'category': 'food'}, 'category_total': 250.0, 'total_spent': 250.0, 'balance': 14750.0}
[step 3] THINK   user mentioned spending 400 on uber ride; log it
          ACT     add_expense({'item': 'uber ride', 'amount': 400.0})
          OBS     {'ok': True, 'recorded': {'item': 'uber ride', 'amount': 400.0, 'category': 'transport'}, 'category_total': 400.0, 'total_spent': 650.0, 'balance': 14350.0}
[step 4] ANSWER  Logged 2 expenses (650 total this session).
----------------------------------------------------------------------
AGENT: Logged 2 expenses (650 total this session).

USER: Also paid 1200 for groceries and a movie cost 300.
[step 

### Where each required behaviour is visible above

| Requirement | Where |
|---|---|
| Calls tools | `ACT add_expense(...)` / `ACT get_summary(...)` lines |
| More than one step | steps 2–4 in turn 1; the affordability turn is `get_summary` → decide |
| Decides next step from a tool result | affordability turn reads `balance` from the `get_summary` OBS, then computes `12850 - 2000` |
| Memory across turns | turn 3 uses the budget from **turn 1** and the expenses from **turns 1–2** |

In [5]:
# Proof the later turn really used earlier memory:
print("expenses remembered:", [(e.item, e.amount, e.category) for e in agent.memory.expenses])
print("budget remembered  :", agent.memory.monthly_budget)
print("total turns in memory:", len(agent.memory.turns))

expenses remembered: [('lunch', 250.0, 'food'), ('uber ride', 400.0, 'transport'), ('groceries', 1200.0, 'food'), ('movie', 300.0, 'entertainment')]
budget remembered  : 15000.0
total turns in memory: 6


## 4. Demo B — memory changes the answer to the *same* question

Ask the affordability question with **no budget known** → the agent inspects the
`get_summary` result, sees `balance is None`, and instead of guessing it **asks for
the budget**. Give the budget, ask again → different plan, real answer.

In [6]:
agent_b = Agent(mode="rule")

for goal in [
    "I spent 500 on shoes and 300 on headphones.",
    "Can I afford a 3000 phone?",                      # no budget yet -> agent asks
    "My budget is 4000. Now can I afford a 3000 phone?",  # now it can decide
]:
    print("\nUSER:", goal)
    r = agent_b.run(goal)
    r.show_trace()
    print("AGENT:", r.answer)


USER: I spent 500 on shoes and 300 on headphones.
[step 1] THINK   user mentioned spending 500 on shoes; log it
          ACT     add_expense({'item': 'shoes', 'amount': 500.0})
          OBS     {'ok': True, 'recorded': {'item': 'shoes', 'amount': 500.0, 'category': 'shopping'}, 'category_total': 500.0, 'total_spent': 500.0, 'balance': None}
[step 2] THINK   user mentioned spending 300 on headphones; log it
          ACT     add_expense({'item': 'headphones', 'amount': 300.0})
          OBS     {'ok': True, 'recorded': {'item': 'headphones', 'amount': 300.0, 'category': 'shopping'}, 'category_total': 800.0, 'total_spent': 800.0, 'balance': None}
[step 3] ANSWER  Logged 2 expenses (800 total this session).
----------------------------------------------------------------------
AGENT: Logged 2 expenses (800 total this session).

USER: Can I afford a 3000 phone?
[step 1] THINK   to judge a 3000 spend I need the balance
          ACT     get_summary({'category': 'all'})
          OBS     

## 5. Demo C — category summary

The agent gathers totals with one `get_summary` call and reports the breakdown
built from everything in memory.

In [7]:
agent_c = Agent(mode="rule")
for goal in [
    "budget is 8000",
    "spent 900 on groceries, 600 on petrol, 250 on coffee and 1500 on rent",
    "where did my money go?",
]:
    print("\nUSER:", goal)
    r = agent_c.run(goal)
    r.show_trace()
    print("AGENT:", r.answer)


USER: budget is 8000
[step 1] MEMORY  stored monthly_budget = 8000
[step 2] ANSWER  I can log expenses and summarise your spending. Try: 'spent 250 on lunch' or 'can I afford a 2000 trip?'.
----------------------------------------------------------------------
AGENT: I can log expenses and summarise your spending. Try: 'spent 250 on lunch' or 'can I afford a 2000 trip?'.

USER: spent 900 on groceries, 600 on petrol, 250 on coffee and 1500 on rent
[step 1] THINK   user mentioned spending 900 on groceries; log it
          ACT     add_expense({'item': 'groceries', 'amount': 900.0})
          OBS     {'ok': True, 'recorded': {'item': 'groceries', 'amount': 900.0, 'category': 'food'}, 'category_total': 900.0, 'total_spent': 900.0, 'balance': 7100.0}
[step 2] THINK   user mentioned spending 600 on petrol; log it
          ACT     add_expense({'item': 'petrol', 'amount': 600.0})
          OBS     {'ok': True, 'recorded': {'item': 'petrol', 'amount': 600.0, 'category': 'transport'}, 'categor

## 6. LLM lane (optional)

Set `GITHUB_TOKEN` (a GitHub Models token) and run the cell below. Same loop, same
tools, same `Memory` — only the *decider* changes: the model emits `tool_calls`,
the loop runs the tool, feeds the result back, and repeats until the model answers.

```python
import os
os.environ["GITHUB_TOKEN"] = "ghp_..."          # your GitHub Models token
llm_agent = Agent(mode="llm")                    # or mode="auto"
r = llm_agent.run("Budget is 12000. Spent 800 on food and 400 on a cab. Can I afford a 2000 trip?")
r.show_trace()
print(r.answer)
```

In [8]:
import os
if os.environ.get("GITHUB_TOKEN"):
    llm_agent = Agent(mode="llm")
    r = llm_agent.run("Budget is 12000. Spent 800 on food and 400 on a cab. Can I afford a 2000 trip?")
    r.show_trace()
    print("AGENT:", r.answer)
else:
    print("GITHUB_TOKEN not set — skipping LLM lane. The rule lane above already "
          "shows the full agent (tools + multi-step + memory).")

GITHUB_TOKEN not set — skipping LLM lane. The rule lane above already shows the full agent (tools + multi-step + memory).
